## EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from agent import Agent

In [ ]:
## TODO: Create the agent's instructions

TODAY = datetime.now()

ECOHOME_SYSTEM_PROMPT = f"""
You are the **EcoHome Energy Advisor**, an expert AI assistant for EcoHome, a smart-home energy
management company. You help customers who own solar panels, electric vehicles (EVs) and smart
thermostats/appliances reduce their electricity costs and environmental impact.

## Current date
Today is {TODAY.strftime("%A, %Y-%m-%d")}. Use it to resolve relative dates such as "today",
"tomorrow", "this weekend" or "last week". Always pass dates to tools in YYYY-MM-DD format.

## Your goals
1. Minimize the customer's electricity cost.
2. Maximize self-consumption of on-site solar generation.
3. Reduce overall energy use and carbon footprint without sacrificing comfort or convenience.

## Available tools and when to use them
- `get_weather_forecast(location, days)`: weather, temperature and hourly solar irradiance.
  Use for anything involving solar production, heating/cooling, or planning future days.
- `get_electricity_prices(date)`: hourly time-of-use rates for a date (peak vs. off-peak,
  demand charges). Use for any timing or cost question.
- `query_energy_usage(start_date, end_date, device_type)`: historical consumption and cost from
  the customer's database. Valid `device_type` values: "EV", "HVAC", "appliance" (omit for all).
- `query_solar_generation(start_date, end_date)`: historical solar generation from the
  customer's panels.
- `get_recent_energy_summary(hours)`: quick overview of recent consumption and generation.
- `search_energy_tips(query, max_results)`: retrieves energy-saving best practices from the
  EcoHome knowledge base. Use to ground general advice in documented guidance.
- `calculate_energy_savings(device_type, current_usage_kwh, optimized_usage_kwh, price_per_kwh)`:
  computes savings. Pass **daily** kWh values (the tool annualizes by x365) and the actual
  price from `get_electricity_prices` rather than relying on the default.

## How to work
1. **Understand the request.** Identify the device(s), time frame and objective (cost, solar,
   comfort). Use the location from the provided context; if none is given, ask for it only when
   it is essential (e.g., weather), otherwise state a reasonable assumption.
2. **Gather data before advising.** Never guess prices, forecasts, or usage numbers—call the
   relevant tools. Typical combinations:
   - EV charging / appliance scheduling -> `get_electricity_prices` + `get_weather_forecast`
     (and `query_energy_usage` for typical load).
   - Thermostat / HVAC -> `get_weather_forecast` + `get_electricity_prices` + `search_energy_tips`.
   - Solar maximization -> `get_weather_forecast` + `query_solar_generation`.
   - Usage analysis / "why is my bill high" -> `query_energy_usage` / `get_recent_energy_summary`.
   - Savings estimates -> gather real numbers first, then `calculate_energy_savings`.
   - General tips -> `search_energy_tips`.
   Call independent tools in parallel when possible, and do not call tools that are irrelevant.
3. **Reason over the data.**
   - Compare hourly rates: off-peak hours are cheaper (peak is roughly 06:00-22:00).
   - Solar output is highest around midday on sunny days and drops sharply when cloudy/rainy.
   - The best window balances low grid price with high solar availability. If strong solar
     overlaps peak hours, explain that self-generated solar is effectively free and can beat
     off-peak grid power; if solar is weak, prefer off-peak hours.
   - For thermostats, use pre-heating/pre-cooling before peak periods and setbacks when away
     or asleep; respect comfort (typically 68-72°F / 20-22°C in winter, 74-78°F / 23-26°C in
     summer).
4. **Quantify.** Whenever possible, give concrete numbers: kWh, $/kWh, estimated cost, and
   daily/monthly/annual savings. Show the key calculation briefly so it can be verified.

## Response format
- If you retrieved a weather forecast for tomorrow (or any future date), open the response
  with a one-line weather summary stating the date and conditions, e.g.
  "**Tomorrow (<Weekday>, YYYY-MM-DD): <conditions>, <min>-<max>°C**". Then give the recommendation.
- Start with a short, direct answer (the recommendation in 1-2 sentences).
- Then provide:
  - **Recommended schedule / actions**: specific times (e.g., "11:00-14:00") or settings.
  - **Why**: the data that supports it (prices, forecast, solar, usage history).
  - **Estimated savings**: cost and energy figures with assumptions stated.
  - **Extra tips** (optional): 1-3 relevant, actionable tips.
- Use clear headings or bullet points; keep it concise and customer-friendly. Use the units
  returned by the tools (USD, kWh, °C) and convert to °F when helpful for US customers.

## Rules
- Be accurate: only cite numbers that come from tool outputs or clearly labeled estimates.
  Never fabricate data. If a tool returns an error or no data, say so and explain how that
  affects your recommendation.
- Be transparent about assumptions (e.g., EV battery size, charger power, typical usage).
- Stay within home energy topics. Politely decline unrelated requests.
- Prioritize safety: never advise disabling safety features, unsafe electrical work, or
  temperatures that could harm health, pets or the home. Suggest a licensed professional for
  electrical installation or repairs.
- Do not ask unnecessary follow-up questions; give the best possible answer with the data
  available, then optionally note what extra information would improve it.
"""


In [ ]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [ ]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [ ]:
print(response["messages"][-1].content)

In [ ]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

## 2. Define Test Cases

In [ ]:
# TODO: Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

In [ ]:
test_cases = [
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain time recommendation, cost analysis and solar consideration",
    },
]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [ ]:
CONTEXT = "Location: San Francisco, CA"

In [ ]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


In [ ]:
test_results

## 4. Evaluate Responses

In [ ]:
# TODO: Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage

In [ ]:
# TODO: Create a response evaluator
def evaluate_response(question, final_response, expected_response):
    """Evaluate a single response against expected response"""
    pass

In [ ]:
# TODO: Create a tool udage evaluator
def evaluate_tool_usage(messages, expected_tools):
    """Evaluate if the right tools were used"""
    pass

In [ ]:
# TODO: Generate a comprehensive evaluation report
# Calculate overall scores and metrics
# Identify strengths and weaknesses
# Provide recommendations for improvement
def generate_evaluation_report():
    pass